In [49]:
import pennylane as qml
from pennylane import numpy as np

In [50]:
A = np.random.randint(1,101,size=(4,4)).astype(float)
b = np.random.randint(1,101,size=(4,1)).astype(float)
B=b
b=b/np.linalg.norm(b)
print(A)
print(B)

[[40.  9. 76. 66.]
 [62.  7. 55. 63.]
 [ 7. 17. 51. 12.]
 [94. 27. 81. 83.]]
[[99.]
 [84.]
 [89.]
 [84.]]


In [51]:
A_b = A@b
print(A_b)

[[ 95.4148572 ]
 [ 94.7927326 ]
 [ 42.97703977]
 [144.34972081]]


In [52]:
norm_a_b = np.linalg.norm(A_b)
x_classical = A_b/norm_a_b
print(x_classical)

[[0.47252727]
 [0.46944629]
 [0.21283712]
 [0.71486958]]


In [53]:
num_qubits = int(np.log2(A.shape[0]))
device = qml.device('default.qubit',wires=num_qubits)
Identity = np.eye(4)
hamiltonian = Identity-(A_b@A_b.conj().T)/norm_a_b**2
hamiltonian = qml.Hermitian(hamiltonian,wires=range(num_qubits))

In [54]:
@qml.qnode(device)
def cost(theta):
    qml.RX(theta[0],wires=0)
    qml.RX(theta[1],wires=1)
    qml.CNOT(wires=[0,1])
    qml.RY(theta[2],wires=0)
    qml.RY(theta[3],wires=1)
    return qml.expval(hamiltonian)

In [55]:
theta = np.random.uniform(0,np.pi,4,requires_grad=True)
optimiser = qml.AdamOptimizer(stepsize=0.2)
for i in range(400):
    theta,costs = optimiser.step_and_cost(cost,theta)
    if i%40==0:
        print(f'theta is {theta} and cost is {costs}')

theta is [1.08985891 1.2553355  2.89792758 0.3663475 ] and cost is 0.8744715879211588
theta is [-0.01843083  0.18680868  1.59381837  1.97321819] and cost is 0.07951685296234826
theta is [0.01009697 0.01366467 1.68326009 2.10263866] and cost is 0.06035682361193635
theta is [-1.25257285e-03  3.45480146e-03  1.69995299e+00  2.12522789e+00] and cost is 0.06021452750813597
theta is [-1.28248308e-04 -3.19820238e-04  1.69906816e+00  2.12455876e+00] and cost is 0.06021234067207848
theta is [-2.21934005e-05  4.47343460e-05  1.69924151e+00  2.12487326e+00] and cost is 0.06021231119217071
theta is [-3.04754108e-07 -4.32154043e-09  1.69925376e+00  2.12493252e+00] and cost is 0.06021230975303531
theta is [ 2.86173571e-07 -1.52573521e-08  1.69925215e+00  2.12492727e+00] and cost is 0.060212309746530364
theta is [-5.26079750e-08 -7.79209769e-08  1.69925247e+00  2.12492761e+00] and cost is 0.06021230974630417
theta is [-2.12617717e-09  1.78670768e-08  1.69925239e+00  2.12492750e+00] and cost is 0.0602

In [56]:
@qml.qnode(device)
def ansatz(theta):
    qml.RX(theta[0],wires=0)
    qml.RX(theta[1],wires=1)
    qml.CNOT(wires=[0,1])
    qml.RY(theta[2],wires=0)
    qml.RY(theta[3],wires=1)
    return qml.state()

In [57]:
quantum_solution = np.real(ansatz(theta))
overlap = np.abs(np.dot(quantum_solution,x_classical))**2
print(f'classical solution is {x_classical*norm_a_b}')
print()
print(f'quantum solution is {quantum_solution.reshape(4,1)*norm_a_b}')
print()
print(f'overlap is {overlap}')

classical solution is [[ 95.4148572 ]
 [ 94.7927326 ]
 [ 42.97703977]
 [144.34972081]]

quantum solution is [[ 64.89139119]
 [116.46571449]
 [ 73.81232891]
 [132.4768273 ]]

overlap is [0.93978769]
